# R3 · 2 · Mistura normal + pico (e revisões mensais)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 e 2):
1. suba **`r3_lib.py`** (pasta `data_imdc_2026/RODADA3/`) e confira que o `r2_lib.py` mais recente está lá;
2. **crie à mão** as pastas `resultados_uf_r3`, `revisoes_r3`, `cache_r3` e `avaliacao_r3`.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r3/uf/` (o CompUF4 e o Comp5 também as enxergam). As revisões de julho a outubro vão para `revisoes_r3/uf/`, fora da régua oficial. As tabelas vão para `avaliacao_r3/`, com data e hora no nome e uma cópia `_ultimo`.

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (variáveis, um arquivo por método × corte do detector, parâmetros em json, um arquivo por modelo × temporada, um PNG por UF) e o que já existe é pulado.

**Dados externos (opcional):** para testar sinais que não estão no FTP (previsão de ENSO do IRI, sorotipo, vacinação, positividade de exames), ponha em `cache_r3/externos_r3.csv` colunas `uf_code`, `season` e, se quiser, `corte` (`jun`/`jul`/`ago`/`set`/`out`), mais uma coluna por sinal. Cada valor deve ser o que se sabia **naquele corte**. Apague `cache_r3/detector_*` e `cache_r3/sinais_detector.json` para refazer.

CPU comum, ~20–40 min. **Rode depois do R3_1.**

- **② normal:** ENS_grafos3_λ, o melhor modelo de temporada da rodada 2, recriado do mesmo jeito que no R2_1 (λ e membros gravados em `avaliacao_r2/`).
- **Trajetórias:** 1.000 trajetórias do normal (coerentes no tempo) e 1.000 sorteios do tamanho previsto pelo detector.
- **Três jeitos de misturar:**
  - `MIX_escala_UF`: quando o tamanho sorteado passa do limiar, a trajetória é reescalada para esse total, mantendo a forma do normal;
  - `MIX_forma_UF`: igual, mas com a forma média dos anos grandes (da UF, senão da macrorregião, senão de todas). Ela já traz o início mais cedo dos anos epidêmicos;
  - `AJ_continuo_UF`: sem limiar; toda trajetória anda uma fração *w* do caminho até o total do detector (*w* = 0 é o normal; *w* = 1 confia só no detector).
- **Escolha dos parâmetros:** o limiar (1,5×, 2× ou 3×) e o *w* (0,25 a 1) são escolhidos pelo menor R × climatologia nas temporadas de afinação (2018–2021), no corte de junho.
- **Revisões:** o mesmo, com o detector de jul/ago/set/out. O normal continua o de junho.


In [ ]:
NOTEBOOK = 'r3_mistura'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r3', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
sys.path.insert(0, PASTA_LOCAL)
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_REV = os.path.join(PASTA_DRIVE, 'revisoes_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV3 = os.path.join(PASTA_DRIVE, 'avaliacao_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R3, 'uf'), os.path.join(PASTA_REV, 'uf'), PASTA_C3, PASTA_AV3):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3
for m in (V, L, CM, U, S, R2, R3):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
def detector_escolhido():
    esc = R3.json_le(os.path.join(PASTA_C3, 'detector_escolhido.json'))
    assert esc, 'Rode antes o R3_1_Detector.'
    P = pd.concat([pd.read_parquet(os.path.join(PASTA_C3, f'detector_{m}_{k}.parquet')) for k, m in esc.items()], ignore_index=True)
    return esc, P


In [ ]:
# @title 2 · Normal (ENS_grafos3_λ), BB e climatologia para afinação + teste (cache) e detector escolhido
NORMAL = R3.normal_ens_grafos3(ctx, PASTA_DRIVE, PASTA_AV2, PASTA_COMP, PASTA_EXT, PASTA_C3, TUNING + TESTE)
ESC, DET = detector_escolhido()
print(NORMAL.groupby(['modelo']).season.unique(), ESC)


In [ ]:
# @title 3 · Parâmetros de cada mistura (escolhidos na afinação; gravados em json)
PAR = R3.escolher_parametros(ctx, NORMAL, DET, PASTA_C3, M=500)
pd.DataFrame({v: dict(parametro=round(p['par'], 3), R_afinacao=min(r for _, r in p['grade']), R_normal_afinacao=round(p['normal'], 3))
              for v, p in PAR.items()}).T


In [ ]:
# @title 4 · Gerar as misturas (jun → resultados_uf_r3; jul–out → revisoes_r3). Um arquivo por modelo × temporada
for v, p in PAR.items():
    for corte in R3.CORTES:
        nome = R3.nome_mistura(v, corte)
        pasta = PASTA_R3 if corte == 'jun' else PASTA_REV
        falta = [s for s in TUNING + TESTE if REFAZER or not os.path.exists(L._arq(pasta, 'uf', nome, s))]
        if not falta:
            continue
        t0 = time.time()
        x = R3.gerar_mistura(ctx, NORMAL, DET, v, p['par'], corte, falta, M=1000)
        if x is None:
            print('sem dados:', nome); continue
        for s, g in x.assign(modelo=nome).groupby('season'):
            g.to_parquet(L._arq(pasta, 'uf', nome, s) + '.tmp', index=False)
            os.replace(L._arq(pasta, 'uf', nome, s) + '.tmp', L._arq(pasta, 'uf', nome, s))
        print(f'✅ {nome:28s} {sorted(x.season.unique())} · {time.time() - t0:.0f}s', flush=True)
